[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jinming99/learn-ml-by-building/blob/main/Lecture%201%20Overview/01-Pre-Lecture-Setup.ipynb)
# Machine Learning Course - Pre-Lecture Verification Notebook

**Run this notebook AFTER completing the terminal setup instructions**


In [ ]:
# @title Colab Setup
import sys
import os

if 'google.colab' in sys.modules:
    print("Detected Google Colab environment.")

    # 1. Clone the repository to access data and utilities
    if not os.path.exists('learn-ml-by-building'):
        print("Cloning repository...")
        !git clone https://github.com/jinming99/learn-ml-by-building.git

    # 2. Change directory to the specific lecture folder
    os.chdir('learn-ml-by-building/Lecture 1 Overview')
    print(f"Changed directory to: {os.getcwd()}")

    # 3. Install required packages
    print("Installing packages...")
    !pip install -q torch transformers accelerate bitsandbytes datasets sentence-transformers numpy pandas scikit-learn matplotlib seaborn tqdm huggingface_hub

    # 4. Handle Secrets (HF_TOKEN)
    from google.colab import userdata
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
        print("✅ HF_TOKEN loaded.")
    except Exception as e:
        print("⚠️ HF_TOKEN not found! Please add 'HF_TOKEN' to the 'Secrets' (key icon) in the sidebar.")

    print("✅ Colab setup complete!")
else:
    print("Running locally. Ensure you have run the setup instructions.")

## System Information


In [ ]:
import sys
import platform
import os
import warnings
warnings.filterwarnings('ignore')

### System Information

In [ ]:
print(f"Python Version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Python Executable: {sys.executable}")

# Check available memory
try:
    import psutil
    memory = psutil.virtual_memory()
    print(f"Available Memory: {memory.available / (1024**3):.2f} GB")
    print(f"Total Memory: {memory.total / (1024**3):.2f} GB")
except ImportError:
    print("Memory check: psutil not installed")

# Check GPU availability
try:
    import torch
    if torch.cuda.is_available():
        print(f"GPU Available: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    else:
        print("GPU Available: CPU only (this is fine for the course!)")
except ImportError:
    print("GPU Check: PyTorch not installed")

## Verify Package Imports

In [ ]:
import importlib

### Verify Package Imports

In [ ]:
packages_to_check = {
    "torch": "PyTorch",
    "transformers": "Transformers",
    "sentence_transformers": "Sentence-Transformers",
    "datasets": "Datasets",
    "numpy": "NumPy",
    "pandas": "Pandas",
    "sklearn": "Scikit-learn",
    "matplotlib": "Matplotlib",
    "seaborn": "Seaborn",
    "tqdm": "TQDM",
    "accelerate": "Accelerate",
}

all_imported = True
for module_name, description in packages_to_check.items():
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "unknown")
        print(f"{description:25s} v{version}")
    except ImportError as e:
        print(f"{description:25s} - Not found")
        all_imported = False

if all_imported:
    print("\nAll packages imported successfully!")
else:
    print("\nSome packages missing. Check terminal setup instructions.")

## Download and Verify Gemma-3-270m Model

Now let's download the Gemma-3-270m model. This is a small model (~550MB) that we'll use in class:

In [ ]:
# Hugging Face authentication is needed only when the local model is missing.
from pathlib import Path

local_dir = (Path.cwd() / "models" / "gemma-3-270m").resolve()
required_model_files = ["config.json", "model.safetensors", "tokenizer.json"]
model_is_ready = all((local_dir / name).is_file() for name in required_model_files)

if model_is_ready:
    print(f"✅ Local Gemma model already present: {local_dir}")
    print("   No Hugging Face login or download is needed.")
else:
    print("Local Gemma model is missing; authenticate before the download cell.")
    print("First accept the license at https://huggingface.co/google/gemma-3-270m")
    from huggingface_hub import login
    login()

In [ ]:
# Explicit local download (no symlinks) + load from local path

from pathlib import Path
from huggingface_hub import snapshot_download  # pip install huggingface_hub
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 1) Choose a lecture-local folder
local_dir = (Path.cwd() / "models" / "gemma-3-270m").resolve()
local_dir.mkdir(parents=True, exist_ok=True)
print("Local model dir:", local_dir)

# 2) Download only when the required local files are absent.
required_model_files = ["config.json", "model.safetensors", "tokenizer.json"]
model_is_ready = all((local_dir / name).is_file() for name in required_model_files)
if not model_is_ready:
    snapshot_download(
        repo_id="google/gemma-3-270m",
        local_dir=str(local_dir),
    )
    print("✅ Files downloaded to:", local_dir)
else:
    print("✅ Model files already present; skipping download.")

# 4) Load from local path
tokenizer = AutoTokenizer.from_pretrained(local_dir, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    local_dir,
    torch_dtype=torch.float32,      # adjust if you have GPU: torch.float16
    low_cpu_mem_usage=True,
)
model.eval()

In [ ]:
# 4) Load from local path
tokenizer = AutoTokenizer.from_pretrained(local_dir, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    local_dir,
    torch_dtype=torch.float32,      # adjust if you have GPU: torch.float16
    low_cpu_mem_usage=True,
)
model.eval()

# 5) Quick smoke test
text = "This is a laptop computer"
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
print("LM logits shape:", outputs.logits.shape)

gen_ids = model.generate(**inputs, max_length=40, do_sample=False)
print("Sample:", tokenizer.decode(gen_ids[0], skip_special_tokens=True))
print("Gemma-3-270m loaded from local directory.")

## Verify WebShop Dataset

In [ ]:
# Verify WebShop "items_ins_v2_1000.json" (attributes/instructions)
import json, os
from collections import Counter

# If running the notebook from "Lecture 1 Overview/", this is correct:
data_path = "data/items_ins_v2_1000.json"
# If running from repo root, use:
# data_path = "Lecture 1 Overview/data/items_ins_v2_1000.json"

if os.path.exists(data_path):
    with open(data_path, "r") as f:
        items = json.load(f)

    print(f"Dataset loaded. Product IDs: {len(items)}")

    # Expect top-level dict keyed by IDs -> dicts with attributes, optional instruction fields
    has_instruction = sum(1 for v in items.values() if "instruction" in v)
    has_attrs = sum(1 for v in items.values() if "attributes" in v and isinstance(v["attributes"], list))
    attr_lengths = [len(v.get("attributes", [])) for v in items.values()]
    avg_attrs = (sum(attr_lengths) / len(attr_lengths)) if attr_lengths else 0.0

    print("\nStats:")
    print(f"  • Entries with attributes: {has_attrs}")
    print(f"  • Avg attributes per entry: {avg_attrs:.2f}")
    print(f"  • Entries with instruction: {has_instruction}")

    # Show a sample entry
    first_key = next(iter(items))
    sample = items[first_key]
    print("\nSample Entry:")
    print(f"  • ID: {first_key}")
    print(f"  • attributes: {sample.get('attributes', [])[:8]}")
    if "instruction" in sample:
        print(f"  • instruction: {sample['instruction'][:80]}...")
        print(f"  • instruction_attributes: {sample.get('instruction_attributes', [])[:8]}")
else:
    print(f"Dataset not found at {data_path}")
    print("   Tip: ensure you run this notebook from 'Lecture 1 Overview/' or fix the relative path.")

In [ ]:
# Verify "items_shuffle_1000.json" (richer fields, may include name/price/etc.)
import json, os
from collections import Counter

path2 = "data/items_shuffle_1000.json"  # or "Lecture 1 Overview/data/..." from repo root
if os.path.exists(path2):
    with open(path2, "r") as f:
        items2 = json.load(f)

    print(f"items_shuffle_1000.json loaded: {len(items2)}")
    # Decide if it's a list or dict
    if isinstance(items2, dict):
        values = items2.values()
    else:
        values = items2

    names = [x.get("name") for x in values if isinstance(x, dict)]
    print("Sample name:", next((n for n in names if n), "N/A"))
else:
    print(f"Not found: {path2}")

## Test ML Components


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sentence_transformers import SentenceTransformer

### Test ML Components

In [ ]:
# Test scikit-learn
X = np.random.rand(100, 10)
y = np.random.randint(0, 3, 100)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
accuracy = knn.score(X_test, y_test)
print(f"Scikit-learn KNN: {accuracy:.2f} accuracy")

# Test sentence transformers
print("\nLoading sentence transformer (first run may take a minute)...")
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
test_sentences = ["laptop computer", "cat toy", "fiction book"]
embeddings = sentence_model.encode(test_sentences)
print(f"Sentence embeddings: shape {embeddings.shape}")

# Test clustering
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=2, random_state=42)
labels = kmeans.fit_predict(embeddings)
print(f"KMeans clustering: {labels}")

# Test matplotlib
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
ax.scatter([1, 2, 3], [1, 4, 2])
ax.set_title("Test Plot")
plt.close()
print("Matplotlib: working")

## Troubleshooting

### Common Issues:

**"Cannot download Gemma model"**
- Accept license at: https://huggingface.co/google/gemma-3-270m
- Run: `huggingface-cli login` in terminal

**"Module not found"**
- Ensure virtual environment is activated
- Re-run package installation from terminal setup

**"Dataset not found"**
- Check file exists in `Lecture 1 Overview/data/`
- Re-run download commands from terminal setup

**"Out of memory"**
- Close other applications
- The setup only needs ~2GB RAM